# Image Similarity Search using Oracle 26ai, Langchain, and OpenCLIP

This notebook demonstrates how to build an image similarity search system using Oracle 26ai's vector storage capabilities, Langchain, and OpenCLIP embeddings.

We'll generate embeddings for a set of training images, store them in Oracle Vector Store, and then search for similar images by comparing their embeddings.

## Install necessary packages

Before running the notebook, ensure you have the following packages installed:

* `langchain-oracledb`: Langchain integration for Oracle databases.
* `langchain-experimental`: Langchain experimental package that includes OpenCLIP support.
* `open_clip_torch`: OpenCLIP model for generating image and text embeddings.
* `pillow`: For image handling.
* `python-dotenv`: To manage environment variables securely.

In [ ]:
!python -m pip install -U langchain-oracledb langchain-experimental open_clip_torch pillow python-dotenv scipy

In [ ]:
import os
import glob

import numpy as np
import oracledb
from dotenv import load_dotenv
from langchain_experimental.open_clip import OpenCLIPEmbeddings
from langchain_oracledb.vectorstores import oraclevs
from langchain_oracledb.vectorstores.oraclevs import OracleVS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document
from PIL import Image
from IPython.display import display

## Setting up the database connection

For this notebook to work, we need to have the following environment variables set in a `.env` file or in your system environment:

* `ORACLE_USERNAME`: Your Oracle database username.
* `ORACLE_PASSWORD`: Your Oracle database password.
* `DSN`: The TNS name for your Oracle database connection.

You also need to store the Oracle Wallet files and reference its location when making the connection. You'll download the wallet from your Oracle Cloud account.

In [ ]:
load_dotenv()

WALLET_DIRECTORY = "../.wallet"
WALLET_PASSWORD = os.getenv("ORACLE_PASSWORD")
DSN = os.getenv("DSN")

connection = oracledb.connect(
    user=os.getenv("ORACLE_USERNAME"),
    password=os.getenv("ORACLE_PASSWORD"),
    dsn=DSN,
    config_dir=WALLET_DIRECTORY,
    wallet_location=WALLET_DIRECTORY,
    wallet_password=WALLET_PASSWORD,
)

## Setting up OpenCLIP embeddings

We'll use OpenCLIP through Langchain's `OpenCLIPEmbeddings` class. OpenCLIP generates embeddings that live in the same vector space for both images and text, enabling cross-modal similarity search.

We'll use the `ViT-H-14` model with the `laion2b_s32b_b79k` checkpoint, which provides a good balance between performance and memory usage.

In [ ]:
clip_embeddings = OpenCLIPEmbeddings(
    model_name="ViT-H-14",
    checkpoint="laion2b_s32b_b79k",
)

## Preparing the training images

We need a directory containing the training images. Update the `IMAGES_DIRECTORY` variable below to point to your set of images.

We'll load all images from the directory and create `Document` objects for each one. The `page_content` of each document will be the image file path (used by OpenCLIP to load and embed the image), and the metadata will store the filename for reference.

In [ ]:
IMAGES_DIRECTORY = "./images/training"

image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]
image_paths = []
for ext in image_extensions:
    image_paths.extend(glob.glob(os.path.join(IMAGES_DIRECTORY, ext)))

image_paths = sorted(image_paths)
print(f"Found {len(image_paths)} images in {IMAGES_DIRECTORY}")

for path in image_paths:
    print(f"  - {os.path.basename(path)}")

## Generating image embeddings and storing them in Oracle

Since `OpenCLIPEmbeddings` uses the `embed_image` method for images (while `OracleVS` expects `embed_documents`), we'll generate the embeddings manually and store them directly in Oracle.

We'll create the vector store table, generate embeddings for each image, and insert them into the database.

In [ ]:
TABLE_NAME = "image_vector_store"

print("Generating image embeddings...")
image_embeddings = clip_embeddings.embed_image(image_paths)

embedding_dim = len(image_embeddings[0])
print(f"Embedding dimension: {embedding_dim}")
print(f"Generated embeddings for {len(image_embeddings)} images.")

## Computing the distance threshold

Let's compute it from the training images themselves. We calculate pairwise cosine distances between all training embeddings and set the threshold to the maximum pairwise distance. Any query image closer than this is "as similar as the training images are to each other."

In [ ]:
from scipy.spatial.distance import pdist

pairwise_distances = pdist(np.array(image_embeddings), metric="cosine")

DISTANCE_THRESHOLD = pairwise_distances.max()

print(f"Pairwise cosine distances across training images:")
print(f"  Min:  {pairwise_distances.min():.4f}")
print(f"  Mean: {pairwise_distances.mean():.4f}")
print(f"  Max:  {pairwise_distances.max():.4f}")
print(f"\nDistance threshold (max pairwise distance): {DISTANCE_THRESHOLD:.4f}")

## Preparing documents for ingestion

We create a `Document` object for each image. The `page_content` stores the filename and the metadata stores the filename and absolute path so we can display results later.

In [ ]:
documents = []
for path in image_paths:
    filename = os.path.basename(path)
    metadata = {"filename": filename, "path": os.path.abspath(path)}
    document = Document(page_content=filename, metadata=metadata)
    documents.append(document)

We need a thin adapter so that `OracleVS` can use our pre-computed image embeddings instead of trying to embed the document text. The adapter returns the image embeddings we already generated during ingestion and delegates to `embed_image` for queries.

In [ ]:
from langchain_core.embeddings import Embeddings


class _ImageEmbeddingAdapter(Embeddings):
    """Wraps OpenCLIPEmbeddings so OracleVS uses pre-computed image vectors."""

    def __init__(self, clip_model, precomputed_embeddings):
        self._clip = clip_model
        self._precomputed = precomputed_embeddings

    def embed_documents(self, texts):
        return self._precomputed

    def embed_query(self, image_uri):
        return self._clip.embed_image([image_uri])[0]


adapter = _ImageEmbeddingAdapter(clip_embeddings, image_embeddings)

In [ ]:
vector_store = OracleVS.from_documents(
    documents,
    adapter,
    client=connection,
    table_name=TABLE_NAME,
    distance_strategy=DistanceStrategy.COSINE,
)

print(f"Successfully stored {len(documents)} image embeddings in Oracle Vector Store.")

## Creating the search index

Now we create an HNSW index for fast approximate nearest neighbor search.

In [ ]:
oraclevs.create_index(
    connection,
    vector_store,
    params={
        "idx_name": "image_vs_hnsw_idx",
        "idx_type": "HNSW",
        "accuracy": 97,
        "parallel": 16,
    },
)

## Searching for similar images

Now we can test our image similarity search. We'll take a query image, generate its embedding using OpenCLIP, and find the most similar images in the vector store.

In [ ]:
def query(image_path):
    print("Query image:")
    display(Image.open(image_path).resize((300, 300)))

    results = vector_store.similarity_search_with_score(image_path, k=3)

    print(f"\nResults (distance threshold: {DISTANCE_THRESHOLD:.4f}):\n")

    filtered = [(doc, score) for doc, score in results if score <= DISTANCE_THRESHOLD]

    if not filtered:
        print("No similar images found below the distance threshold.")
    else:
        for i, (doc, score) in enumerate(filtered):
            print(f"{i + 1}. {doc.metadata['filename']} (distance: {score:.4f})")
            result_image_path = doc.metadata["path"]
            if os.path.exists(result_image_path):
                display(Image.open(result_image_path).resize((300, 300)))
            print()

In [ ]:
query("./images/query/sample.png")